In [4]:
import hail as hl
#import argparse
#import pandas
import os

from ukb_utils import hail_init
from ukb_utils import genotypes
from ukb_utils import samples
from ko_utils import qc
from ko_utils import analysis

In [5]:
os.chdir('/well/lindgren/UKBIOBANK/flassen/projects/KO/wes_ko_ukbb/')
hail_init.hail_bmrc_init('logs/hail/hail_test_export.log', 'GRCh38')

2021-11-09 14:13:35 Hail: WARN: Hail has already been initialized. If this call was intended to change configuration, close the session with hl.stop() first.


FatalError: IllegalArgumentException: requirement failed

Java stack trace:
java.lang.IllegalArgumentException: requirement failed
	at scala.Predef$.require(Predef.scala:268)
	at is.hail.backend.spark.SparkBackend$.apply(SparkBackend.scala:218)
	at is.hail.backend.spark.SparkBackend.apply(SparkBackend.scala)
	at sun.reflect.NativeMethodAccessorImpl.invoke0(Native Method)
	at sun.reflect.NativeMethodAccessorImpl.invoke(NativeMethodAccessorImpl.java:62)
	at sun.reflect.DelegatingMethodAccessorImpl.invoke(DelegatingMethodAccessorImpl.java:43)
	at java.lang.reflect.Method.invoke(Method.java:498)
	at py4j.reflection.MethodInvoker.invoke(MethodInvoker.java:244)
	at py4j.reflection.ReflectionEngine.invoke(ReflectionEngine.java:357)
	at py4j.Gateway.invoke(Gateway.java:282)
	at py4j.commands.AbstractCommand.invokeMethod(AbstractCommand.java:132)
	at py4j.commands.CallCommand.execute(CallCommand.java:79)
	at py4j.GatewayConnection.run(GatewayConnection.java:238)
	at java.lang.Thread.run(Thread.

In [ ]:
ht1 = hl.read_table('data/qc/ukb_wes_200k_chr21_variants_phased.ht')
ht1 = ht1.annotate(phased=1)
ht2 = hl.read_table('data/qc/ukb_wes_200k_chr21_variants_unphased.ht')
ht2 = ht2.annotate(phased=0)
#ht = ht1.union(ht2)

In [ ]:
print(ht1.count())
print(ht2.count())
print(ht.count())

In [ ]:
ht1.describe()

In [ ]:
ht2.describe()

In [7]:
chrom = 21
input_phased_path='/well/lindgren/UKBIOBANK/nbaya/wes_200k/phase_ukb_wes/data/phased/non_singleton/ukb_wes_phased_non_singleton_chr21-24xlong.qc-v4.2.2.vcf.gz'
input_unphased_path='data/unphased/post-qc/ukb_wes_200k_filtered_chr21.mt'

input_gnomad_path='/well/lindgren/flassen/ressources/gnomad/gnomad_v2_liftover/exomes/gnomad.exomes.r2.1.1.sites.21.liftover_grch38.vcf.bgz'
input_imputed_path='/well/lindgren/UKBIOBANK/flassen/projects/ukb_compare/data/imputed/GRCh38/ukb_imp_liftover_chr21.mt'

input_annotation_path='data/vep/hail/new/ukb_wes_200k_chr21_vep.ht'



In [12]:
 # get tables
mt1 = qc.get_table(input_path=input_phased_path, input_type='vcf') # 12788
mt2 = qc.get_table(input_path=input_unphased_path, input_type='mt') # 11867 (for singletons)

    # remove withdrawn samples
mt1 = samples.remove_withdrawn(mt1)
mt2 = samples.remove_withdrawn(mt2)

if True:
    gnomad_variants_ht = hl.import_vcf(input_gnomad_path, reference_genome ='GRCh38', force_bgz=True, array_elements_required=False).rows()
    mt1 = mt1.annotate_rows(inGnomAD = hl.is_defined(gnomad_variants_ht[mt1.row_key]))
    mt2 = mt2.annotate_rows(inGnomAD = hl.is_defined(gnomad_variants_ht[mt2.row_key]))
    
# annotate with imputed data
if True:
    imputed = hl.read_table(input_imputed_path)
    mt1 = mt1.annotate_rows(imputed_info = imputed[mt1.row_key].info) 
    mt2 = mt2.annotate_rows(imputed_info = imputed[mt2.row_key].info)
    
# add annotations from table
consequence_annotations = hl.read_table(input_annotation_path)
mt1 = mt1.annotate_rows(consequence = consequence_annotations[mt1.row_key]) 
mt2 = mt2.annotate_rows(consequence = consequence_annotations[mt2.row_key]) 

# annotate entries with DP and DQ
mt1 = mt1.annotate_entries(DP = mt2[(mt1.locus, mt1.alleles), mt1.s].DP)
mt1 = mt1.annotate_entries(GQ = mt2[(mt1.locus, mt1.alleles), mt1.s].GQ)

2021-11-09 14:24:45 Hail: INFO: Reading table without type imputation
  Loading field 'f0' as type str (user-supplied)
2021-11-09 14:24:45 Hail: INFO: Reading table without type imputation
  Loading field 'f0' as type str (user-supplied)


In [21]:
# Perform variant QC on phased data
mt1 = hl.variant_qc(mt1, name='variant_qc')
mt1_rows = mt1.rows().select('variant_qc','imputed_info', 'inGnomAD')

# Perform variant QC on singletons
mt2 = mt2.filter_rows(mt2.info.AC==1)
mt2 = hl.variant_qc(mt2, name='variant_qc')
mt2_rows = mt2.rows().select('variant_qc','imputed_info', 'inGnomAD')

# Combine the data
mt_rows = mt1_rows.union(mt2_rows)
mt_rows = mt_rows.annotate(worst_csq_for_variant_canonical = consequence_annotations[mt_rows.key].vep.worst_csq_for_variant_canonical)
mt_rows.write(out_prefix + "_variants_qc.ht", overwrite=True)
mt_rows.flatten().export(out_prefix + "_variants_qc.vcf.bgz")


In [23]:
mt_rows.describe()

----------------------------------------
Global fields:
    None
----------------------------------------
Row fields:
    'locus': locus<GRCh38> 
    'alleles': array<str> 
    'variant_qc': struct {
        dp_stats: struct {
            mean: float64, 
            stdev: float64, 
            min: float64, 
            max: float64
        }, 
        gq_stats: struct {
            mean: float64, 
            stdev: float64, 
            min: float64, 
            max: float64
        }, 
        AC: array<int32>, 
        AF: array<float64>, 
        AN: int32, 
        homozygote_count: array<int32>, 
        call_rate: float64, 
        n_called: int64, 
        n_not_called: int64, 
        n_filtered: int64, 
        n_het: int64, 
        n_non_ref: int64, 
        het_freq_hwe: float64, 
        p_value_hwe: float64
    } 
    'imputed_info': float64 
    'inGnomAD': bool 
    'worst_csq_for_variant_canonical': struct {
        allele_num: int32, 
        amino_acids: str, 
  

In [220]:
 # get tables
mt1 = qc.get_table(input_path=input_phased_path, input_type='vcf') # 12788
mt2 = qc.get_table(input_path=input_unphased_path, input_type='mt') # 11867 (for singletons)
mt2 = mt2.filter_rows(mt2.info.AC > 1)

In [221]:
final_variant_list = '/well/lindgren/UKBIOBANK/dpalmer/wes_200k/ukb_wes_qc/data/variants/08_final_qc.keep.variant_list'

In [222]:
mt1 = qc.get_table(input_path=input_phased_path, input_type='vcf') # 12788
mt2 = qc.get_table(input_path=input_unphased_path, input_type='mt') # 11867 (for singletons)

In [223]:
#mt1 = mt1.filter_cols(hl.is_defined(ht_final_samples[mt1.col_key]))
#mt2 = mt2.filter_cols(hl.is_defined(ht_final_samples[mt2.col_key]))

In [224]:
ht_final_variants = hl.import_table(final_variant_list, types={'locus':hl.tlocus(reference_genome='GRCh38'), 'alleles':hl.tarray(hl.tstr)},)
ht_final_variants = ht_final_variants.key_by(ht_final_variants.locus, ht_final_variants.alleles)
mt1 = mt1.filter_rows(hl.is_defined(ht_final_variants[mt1.row_key]))
mt2 = mt2.filter_rows(hl.is_defined(ht_final_variants[mt2.row_key]))

2021-11-09 18:00:11 Hail: INFO: Reading table without type imputation
  Loading field 'locus' as type locus<GRCh38> (user-supplied)
  Loading field 'alleles' as type array<str> (user-supplied)


In [225]:
#x = mt1.s.collect()
#y = mt2.s.collect()

In [226]:
#print(len(x))
#print(len(y))

In [227]:
#mt1 = samples.remove_withdrawn(mt1)
#mt2 = samples.remove_withdrawn(mt2)

In [228]:
#x = mt1.s.collect()
#y = mt2.s.collect()

In [229]:
#print(len(x))
#print(len(y))

In [230]:
#mt2 = mt2.filter_rows(mt2.info.AC > 1)

In [ ]:
x = mt1.locus.position.collect()
y = mt2.locus.position.collect()

2021-11-09 18:10:59 Hail: INFO: Coerced prefix-sorted dataset
2021-11-09 18:11:04 Hail: INFO: Coerced prefix-sorted dataset
2021-11-09 18:11:23 Hail: INFO: Ordering unsorted dataset with network shuffle
2021-11-09 18:11:30 Hail: INFO: Ordering unsorted dataset with network shuffle
2021-11-09 18:14:46 Hail: INFO: Ordering unsorted dataset with network shuffle


In [ ]:
print(len(x))
print(len(y))

In [200]:
ht_final_variants = hl.import_table(final_sample_list, no_header=True, key='f0', delimiter=',')
ht_final_samples = hl.import_table(final_sample_list, no_header=True, key='f0', delimiter=',')

2021-11-09 17:58:40 Hail: INFO: Loading 127 fields. Counts by type:
  str: 127
2021-11-09 17:58:40 Hail: INFO: Loading 127 fields. Counts by type:
  str: 127


In [60]:
final = ht_final_samples.f0.collect()

In [61]:
len(final)

176935

In [173]:
final_sample_list='/well/lindgren/UKBIOBANK/dpalmer/wes_200k/ukb_wes_qc/data/samples/09_final_qc.keep.sample_list'
final_variant_list='/well/lindgren/UKBIOBANK/dpalmer/wes_200k/ukb_wes_qc/data/samples/09_final_qc.keep.sample_list'

In [42]:
# filter by final samples
if final_sample_list:
    ht_final_samples = hl.import_table(final_sample_list, no_header=True, key='f0', delimiter=',')
    mt = mt.filter_cols(hl.is_defined(ht_final_samples[mt.col_key]))
   
# filter by final variants
if final_variant_list:
    ht_final_variants = hl.import_table(final_variant_list, types={'locus':hl.tlocus(reference_genome='GRCh38'), 'alleles':hl.tarray(hl.tstr)})
    ht_final_variants = ht_final_variants.key_by(ht_final_variants.locus, ht_final_variants.alleles)
    mt = mt.filter_rows(hl.is_defined(ht_final_variants[mt.row_key]))


2021-11-09 14:59:44 Hail: INFO: Loading 127 fields. Counts by type:
  str: 127


NameError: name 'mt' is not defined

In [39]:
print(mt1.count())
print(mt2.count())

2021-11-09 14:42:54 Hail: INFO: Coerced prefix-sorted dataset
2021-11-09 14:44:39 Hail: INFO: Coerced prefix-sorted dataset


(60590, 185506)
(139596, 199795)


In [36]:
# subset to unphased singletons and combine the two files
mt2 = mt2.filter_rows(mt2.info.AC == 1)
mt2 = mt2.drop('info').select_entries('GT')
mt1 = mt1.drop('info')
mt = mt1.union_rows(mt2)

2021-11-09 14:40:30 Hail: INFO: Coerced prefix-sorted dataset


ValueError: 'MatrixTable.union_rows' expects all datasets to have the same columns. Datasets 0 and 1 have different columns (or possibly different order).

----------------------------------------
Global fields:
    None
----------------------------------------
Column fields:
    's': str
----------------------------------------
Row fields:
    'locus': locus<GRCh38>
    'alleles': array<str>
    'rsid': str
    'qual': float64
    'filters': set<str>
----------------------------------------
Entry fields:
    'GT': call
----------------------------------------
Column key: ['s']
Row key: ['locus', 'alleles']
----------------------------------------


In [ ]:
 # write out variant stats
    #print(f'chr{chrom}: Writing out variants stats to {out_prefix}_variants*')
    #mt1 = hl.variant_qc(mt1, name='variant_qc')
    #ht1_rows_filter = mt1.rows()
    #ht1_rows_filter.write(out_prefix + "_variants_phased.ht", overwrite=True)
    #ht1_rows_filter.flatten().export(out_prefix + "_variants_phased.vcf.bgz")

    #mt2 = hl.variant_qc(mt2, name='variant_qc')
    #ht2_rows_filter = mt2.rows()
    #ht2_rows_filter.write(out_prefix + "_variants_unphased.ht", overwrite=True)
    #ht2_rows_filter.flatten().export(out_prefix + "_variants_unphased.vcf.bgz")

    # annotate consequence category
    category_annotation_mt2 = analysis.annotation_case_builder(mt2.consequence.vep.worst_csq_for_variant_canonical, use_loftee = True)
    category_annotation_mt1 = analysis.annotation_case_builder(mt1.consequence.vep.worst_csq_for_variant_canonical, use_loftee = True)
    mt2 = mt2.annotate_rows(consequence_category = category_annotation_mt2)
    mt1 = mt1.annotate_rows(consequence_category = category_annotation_mt1)

    # write out summary stats
    #summary_count_urv(mt1).export(out_prefix + "_variants_summary_phased.tsv.bgz")
    #summary_count_urv(mt1).export(out_prefix + "_variants_summary_unphased.tsv.bgz")
    
    # get homozygous stats
    #summary_count_homozygous_urv(mt1).export(out_prefix + "_variants_homozygous_summary_phased.tsv.bgz")
    #summary_count_homozygous_urv(mt2).export(out_prefix + "_variants_homozygous_summary_unphased.tsv.bgz")

    # explode rows by gene
    mt1 = mt1.explode_rows(mt1.consequence.vep.worst_csq_by_gene_canonical)
    mt2 = mt2.explode_rows(mt2.consequence.vep.worst_csq_by_gene_canonical)

    # count URV by genes
    mt1_genes = analysis.count_urv_by_genes(mt1, mt1.consequence.vep.worst_csq_for_variant_canonical.gene_id)
    mt1_genes.entries().flatten().export(out_prefix + "_urv_by_genes_phased.tsv.bgz")
    mt2_genes = analysis.count_urv_by_genes(mt2, mt2.consequence.vep.worst_csq_for_variant_canonical.gene_id)
    mt2_genes.entries().flatten().export(out_prefix + "_urv_by_genes_unphased.tsv.bgz")


In [143]:
 # get tables
mt1 = qc.get_table(input_path=input_phased_path, input_type='vcf') # 12788
mt2 = qc.get_table(input_path=input_unphased_path, input_type='mt') # 11867 (for singletons)

<StringExpression of type str>

<StringExpression of type str>

In [197]:
mt = hl.read_matrix_table('data/mt/ukb_wes_200k_annotated_chr21.mt/')

In [198]:
mt.count()

(60590, 185506)